# Experiment 45 — SparseWalker credit-radius ladder

Where does SparseWalker actually need gradient credit?

All arms use the same v1.1 architecture, the same initialization, FullCE objective, AdamW, ML-1M leave-two-out protocol, and full-catalog validation. The only change is where recurrent state is detached.

- `readout_only`: routing/hops detached; gradient only through current readout
- `event_local`: previous state detached every event; gradient through current router + two hops + readout
- `tbptt2`, `tbptt4`, `tbptt8`: exact gradient across 2/4/8 events
- `full_bptt`: no temporal detach inside the sampled window

No test-set evaluation is used during this screening run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, json, torch
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='research/active'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
for p in [f'{REPO}/src', f'{REPO}/experiments']:
    if p not in sys.path: sys.path.insert(0,p)
HEAD=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
import sparsewalker
print('IMPORT_OK', sparsewalker.__file__, flush=True)
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,flush=True)
print('TORCH',torch.__version__,flush=True)
print('HEAD',HEAD,flush=True)
assert torch.cuda.is_available(), 'GPU runtime required'


## Run the ladder

Default is 5 epochs per arm. This is a diagnostic, not a tuned benchmark. `progress_every=0` keeps output readable; every epoch still prints its validation result.


In [ ]:
import runpy, os, sys
EPOCHS_PER_ARM=5
SCRIPT=f'{REPO}/experiments/run_ml1m_credit_radius_ladder.py'
assert Path(SCRIPT).exists(), SCRIPT
text=Path(SCRIPT).read_text()
assert 'Experiment 45' in text and 'event_local' in text and 'full_bptt' in text
argv=[SCRIPT,
      '--epochs-per-arm',str(EPOCHS_PER_ARM),
      '--batch-size','128',
      '--eval-batch-size','1024',
      '--progress-every','0',
      '--data-dir','/content/drive/MyDrive/sparsewalker_data']
print('RUNNING_IN_PROCESS',' '.join(argv),flush=True)
old_argv=sys.argv[:]; old_cwd=os.getcwd()
sys.argv=argv; os.chdir(REPO)
try:
    runpy.run_path(SCRIPT,run_name='__main__')
finally:
    sys.argv=old_argv; os.chdir(old_cwd)


## Result

The key curve is NDCG versus temporal credit radius. If `event_local` is already strong, we do not need BPTT; we need a good way to replace the gradient through one sparse event.


In [ ]:
import pandas as pd, json
root=Path('/content/drive/MyDrive/sparsewalker_credit_radius/seed42')
summary=json.loads((root/'summary.json').read_text())
df=pd.DataFrame(summary['ranking'])
display(df[['arm','credit_radius','best_epoch','best_val_NDCG@10','final_val_NDCG@10','mean_positions_per_s','mean_backward_calls','final_grad_router','final_grad_graph','final_grad_concept_key']])
print(json.dumps(summary,indent=2))
